In [ ]:
import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("✅ All libraries imported successfully!")

In [ ]:
pip install matplotlib

In [ ]:
!pip install seaborn

In [ ]:

df_fake = pd.read_csv('Fake.csv')
df_true = pd.read_csv('True.csv')

# Assign labels: 0 = Fake, 1 = Real
df_fake['label'] = 0
df_true['label'] = 1

print(f"Fake news articles : {df_fake.shape[0]}")
print(f"True news articles : {df_true.shape[0]}")

In [ ]:
df_fake.head(3)

In [ ]:
df_true.head(3)

In [ ]:
df = pd.concat([df_fake, df_true], axis=0)


df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total articles : {df.shape[0]}")
print(f"Columns        : {list(df.columns)}")
print(f"\nNull values:\n{df.isnull().sum()}")

In [ ]:
plt.figure(figsize=(5, 4))
ax = sns.countplot(x='label', data=df, palette=['#e74c3c', '#2ecc71'])
ax.set_xticklabels(['Fake (0)', 'Real (1)'])
plt.title('Class Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [9]:
def clean_text(text):
    """Clean and normalize news text."""
    text = str(text).lower()                                     
    text = re.sub(r'\[.*?\]', '', text)                          
    text = re.sub(r'https?://\S+|www\.\S+', '', text)           
    text = re.sub(r'<.*?>', '', text)                          
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)  
    text = re.sub(r'\n', ' ', text)                              
    text = re.sub(r'\w*\d\w*', '', text)                        
    text = re.sub(r'\s+', ' ', text).strip()                    
    return text


df['content'] = (df['title'].fillna('') + ' ' + df['text'].fillna(''))
df['content'] = df['content'].apply(clean_text)

print("✅ Text cleaned!")
print("\nSample cleaned text:")
print(df['content'].iloc[0][:300])

✅ Text cleaned!

Sample cleaned text:
ben stein calls out circuit court committed a ‘coup d’état’ against the constitution century wire says ben stein reputable professor from pepperdine university also of some hollywood fame appearing in tv shows and films such as ferris bueller s day off made some provocative statements on judge jeani


In [10]:
X = df['content']
y = df['label']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)


vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), stop_words='english')

Xv_train = vectorizer.fit_transform(X_train)   
Xv_test  = vectorizer.transform(X_test)         

print(f"Training samples : {Xv_train.shape[0]}")
print(f"Testing  samples : {Xv_test.shape[0]}")
print(f"Feature size     : {Xv_train.shape[1]}")

Training samples : 33673
Testing  samples : 11225
Feature size     : 50000


In [11]:
LR = LogisticRegression(max_iter=1000, random_state=42)
LR.fit(Xv_train, y_train)

pred_lr = LR.predict(Xv_test)

print(f"Logistic Regression Accuracy : {accuracy_score(y_test, pred_lr)*100:.2f}%")
print()
print(classification_report(y_test, pred_lr, target_names=['Fake', 'Real']))

Logistic Regression Accuracy : 98.65%

              precision    recall  f1-score   support

        Fake       0.99      0.98      0.99      5863
        Real       0.98      0.99      0.99      5362

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [12]:
DT = DecisionTreeClassifier(random_state=42)
DT.fit(Xv_train, y_train)

pred_dt = DT.predict(Xv_test)

print(f"Decision Tree Accuracy : {accuracy_score(y_test, pred_dt)*100:.2f}%")
print()
print(classification_report(y_test, pred_dt, target_names=['Fake', 'Real']))

Decision Tree Accuracy : 99.55%

              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      5863
        Real       1.00      0.99      1.00      5362

    accuracy                           1.00     11225
   macro avg       1.00      1.00      1.00     11225
weighted avg       1.00      1.00      1.00     11225



In [ ]:
GB = GradientBoostingClassifier(random_state=42)
GB.fit(Xv_train, y_train)

pred_gb = GB.predict(Xv_test)

print(f"Gradient Boosting Accuracy : {accuracy_score(y_test, pred_gb)*100:.2f}%")
print()
print(classification_report(y_test, pred_gb, target_names=['Fake', 'Real']))

In [ ]:
RF = RandomForestClassifier(n_estimators=100, random_state=42)
RF.fit(Xv_train, y_train)

pred_rf = RF.predict(Xv_test)

print(f"Random Forest Accuracy : {accuracy_score(y_test, pred_rf)*100:.2f}%")
print()
print(classification_report(y_test, pred_rf, target_names=['Fake', 'Real']))

In [ ]:
results = {
    'Logistic Regression'      : accuracy_score(y_test, pred_lr),
    'Decision Tree'            : accuracy_score(y_test, pred_dt),
    'Gradient Boosting'        : accuracy_score(y_test, pred_gb),
    'Random Forest'            : accuracy_score(y_test, pred_rf),
}

results_df = pd.DataFrame(results.items(), columns=['Model', 'Accuracy'])
results_df['Accuracy (%)'] = (results_df['Accuracy'] * 100).round(2)
results_df = results_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)

print(results_df[['Model', 'Accuracy (%)']].to_string(index=False))

# Bar chart
plt.figure(figsize=(8, 4))
colors = ['#2ecc71' if acc == results_df['Accuracy'].max() else '#3498db' for acc in results_df['Accuracy']]
bars = plt.bar(results_df['Model'], results_df['Accuracy (%)'], color=colors, edgecolor='white')
plt.ylim(80, 100)
plt.ylabel('Accuracy (%)')
plt.title('Model Accuracy Comparison')
plt.xticks(rotation=15, ha='right')
for bar, val in zip(bars, results_df['Accuracy (%)']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{val}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Use Logistic Regression as example (typically best for text)
cm = confusion_matrix(y_test, pred_lr)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Fake', 'Predicted Real'],
            yticklabels=['Actual Fake', 'Actual Real'])
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

In [ ]:
def output_label(n):
    return "🟢 Real News" if n == 1 else "🔴 Fake News"

def manual_testing(news_text):
    """Predict whether a given news article is Fake or Real."""
    
    cleaned = clean_text(news_text)
    
    
    vec_input = vectorizer.transform([cleaned])
    
    
    print("="*50)
    print("📰 NEWS PREDICTION RESULTS")
    print("="*50)
    print(f"Logistic Regression  : {output_label(LR.predict(vec_input)[0])}")
    print(f"Decision Tree        : {output_label(DT.predict(vec_input)[0])}")
    print(f"Gradient Boosting    : {output_label(GB.predict(vec_input)[0])}")
    print(f"Random Forest        : {output_label(RF.predict(vec_input)[0])}")
    print("="*50)

print("✅ manual_testing() function is ready!")

In [ ]:

news = str(input("Enter news article text: "))
manual_testing(news)

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr


def output_label(n):
    return "🟢 Real News" if n == 1 else "🔴 Fake News"


def predict_news(news_text):
    if not news_text.strip():
        return "⚠️ Please enter some news text.", "", "", "", ""

    cleaned    = clean_text(news_text)
    vec_input  = vectorizer.transform([cleaned])

    lr_pred  = LR.predict(vec_input)[0]
    dt_pred  = DT.predict(vec_input)[0]
    gb_pred  = GB.predict(vec_input)[0]
    rf_pred  = RF.predict(vec_input)[0]

    votes    = [lr_pred, dt_pred, gb_pred, rf_pred]
    verdict  = "🟢 REAL NEWS" if sum(votes) >= 2 else "🔴 FAKE NEWS"
    confidence = f"{(sum(votes) / 4) * 100:.0f}% models agree"

    summary = f"""
## {verdict}
**{confidence}**
"""
    return (
        summary,
        output_label(lr_pred),
        output_label(dt_pred),
        output_label(gb_pred),
        output_label(rf_pred),
    )


from sklearn.metrics import accuracy_score

acc_lr = f"{accuracy_score(y_test, pred_lr)*100:.2f}%"
acc_dt = f"{accuracy_score(y_test, pred_dt)*100:.2f}%"
acc_gb = f"{accuracy_score(y_test, pred_gb)*100:.2f}%"
acc_rf = f"{accuracy_score(y_test, pred_rf)*100:.2f}%"

# Gradio UI 
with gr.Blocks(title="Fake News Detector", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 📰 Fake News Detection System
    ### Harsh Raj | Amity University Jharkhand
    Paste any news article below and all 4 ML models will predict whether it's **Fake** or **Real**.
    ---
    """)

    with gr.Row():
        with gr.Column(scale=2):
            news_input = gr.Textbox(
                label="📋 Paste News Article Here",
                placeholder="Enter any news article text...",
                lines=8
            )
            predict_btn = gr.Button("🔍 Analyze News", variant="primary", size="lg")
            clear_btn   = gr.Button("🗑 Clear", variant="secondary")

        with gr.Column(scale=1):
            gr.Markdown("### 🤖 Model Accuracy (Test Set)")
            gr.DataFrame(
                value=[
                    ["Logistic Regression", acc_lr],
                    ["Decision Tree",       acc_dt],
                    ["Gradient Boosting",   acc_gb],
                    ["Random Forest",       acc_rf],
                ],
                headers=["Model", "Accuracy"],
                interactive=False,
            )

    gr.Markdown("---")
    gr.Markdown("### 📊 Prediction Results")

    verdict_box = gr.Markdown("*Results will appear here...*")

    with gr.Row():
        lr_out = gr.Textbox(label="Logistic Regression", interactive=False)
        dt_out = gr.Textbox(label="Decision Tree",       interactive=False)
        gb_out = gr.Textbox(label="Gradient Boosting",   interactive=False)
        rf_out = gr.Textbox(label="Random Forest",       interactive=False)

    
    gr.Markdown("---")
    gr.Markdown("### 🧪 Try a Sample")
    with gr.Row():
        gr.Examples(
            examples=[
                ["Scientists have developed a new vaccine that shows 95% effectiveness against the latest flu strain in clinical trials conducted across 12 countries."],
                ["SHOCKING: Government puts mind control chips in COVID vaccines! Whistleblower reveals secret plan to track all citizens via 5G towers."],
                ["The Federal Reserve raised interest rates by 0.25% today in an effort to combat rising inflation, as reported by major financial outlets."],
            ],
            inputs=news_input,
            label="Click a sample to load it"
        )

    
    predict_btn.click(
        fn=predict_news,
        inputs=news_input,
        outputs=[verdict_box, lr_out, dt_out, gb_out, rf_out]
    )
    clear_btn.click(
        fn=lambda: ("*Results will appear here...*", "", "", "", ""),
        inputs=None,
        outputs=[verdict_box, lr_out, dt_out, gb_out, rf_out]
    )

demo.launch(share=True)   # share=True gives you a public link valid for 72 hrs